## **Finetune BERT Encoder for Text Classification**

#### **0. Housekeeping Steps**

Let us perform the necessary housekeeping steps before procedding further with the Machine Learning task at hand. We probably need to install some packages before we can import them.

In [ ]:
!pip install transformers
!pip install -U datasets

In addition we need to link our Colab notebook to our Google Drive so we can both save and load our necessary models and data. To do this we need to mount our Google Drive locally. Please be mindful of repeating this step.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
#sanity check to ensure drive was properly mounted
!ls /content/gdrive/MyDrive/DSSI-2024-Week2-internal/notebooks

 Day1-rough.ipynb
 e2e-dataset
'Homework1+2: Text Classification with Pretrained Embeddings and Finetuning.ipynb'
'Lab 1+2: PyTorch+Neural-Networks-for-Beginners.ipynb'
'Lab 3: Classification for NLP Demo.ipynb'
'Lab 4: Hugging_Face_Transformers_Tutorial'
 sample_hf_trainer
 sst-model


**NOTE:** In order to load and save models we need to set the model path. First, create a folder named `sst-model` in this path where you can save your best performing model.

In [ ]:
model_path = "/content/gdrive/MyDrive/DSSI-2024-Week2-internal/notebooks/sst-model/"

#### **1. Imports**

Let us start with all the necessary imports

In [ ]:
from collections import defaultdict, Counter
import json
import numpy as np
import torch

from matplotlib import pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, DatasetDict
from torch.utils.data import DataLoader

#### **2. Loading the data**

We start with loading a dataset from the Hugging Face hub. Note that for our projects we will have to work with custom datasets and that would require extending the `Dataset` class to a custom Dataset subclass specific to your dataset. However, in the interest of time, we will simply download an existing dataset from the hub that is already in the desired format.

For the purpose of this homework, we want to use the **Stanford Sentiment Treebank Dataset** for sentiment classification.

In [ ]:
#Download the Stanford Sentiment Treebank Dataset

dataset_name = "stanfordnlp/sst2"
sst_dataset = load_dataset(dataset_name)

sst_dataset

#### **3. Dataset Preprocessing**

Next we tokenize and prepare the data. For our choice of model and associated tokenizer we want to use `bert-base-uncased` from the Hugging Face model hub.

As you might remember, the tokenizer executes the following steps:


1.   Split text into tokens and convert them into word ids
2.   Add special tokens like [CLS] and [SEP]
3.   Padding the text so all inputs are of the same length
4.   Apply truncation when needed by setting a max length for sequences.

***Initialize your tokenizer here and call it to perform the aforementioned pre-processing.***

In [ ]:
# from transformers import BertTokenizer, BertModel, BertConfig, BertForSequenceClassification
# name = "google-bert/bert-base-cased"

from transformers import DistilBertConfig, DistilBertTokenizer, DistilBertForSequenceClassification, DistilBertModel
name = "distilbert/distilbert-base-cased"

#Initialize your tokenizer here
tokenizer = <--->

sample_input = "We want to use a pretrained tokenizer."

#Call your tokenizer here to check if it was properly loaded by using on a test sentence
tokenized_inputs = tokenizer(
    <--->
)
print(tokenized_inputs["input_ids"])

Now that our tokenizer has been loaded let us use it to tokenize our entire dataset. We will use the function that we use to test the tokenizer on a single input. We will also split our data into batches of 128.

In [ ]:
# Now that our tokenizer has been properly loaded, we need to call the tokenizer
# for every example in the dataset. Here we use list comprehension with a
# lambda function ensure that.

tokenized_sst_dataset = sst_dataset.map(
    lambda example: tokenizer(example['sentence'], padding="max_length",
    truncation=True, max_length=64)
)

# We need to remove these extra columns before the dataset can be sent to the
# dataloader and subsequently to the model. Also be sure to check that the
# output column is named labels or else rename if necessary
tokenized_sst_dataset = tokenized_sst_dataset.remove_columns(['idx', 'sentence'])
tokenized_sst_dataset = tokenized_sst_dataset.rename_column("label", "labels")
tokenized_sst_dataset.set_format("torch")

In [ ]:
#lets check our tokenization for a few samples
tokenized_sst_dataset['train'][0:2]

#### **4. Using DataLoader to batchify data**

We need to send our datasets to the Dataloader in order to segment the data into batches.

Remember that we need batches to run our iterative optimization procedure which is typically some form of Mini-batch Gradient Descent.

In the interest of time we first sample 2048 records out of the training set and train the model of this reduced sample instead of the entire 62K sample size.

***Please send your reduced training dataset and the validation dataset to the DataLoader to get the corresponding training and validation dataloaders.***

In [ ]:
train_dataset = tokenized_sst_dataset['train'].shuffle(seed=1111).select(range(2048))
train_dataloader = DataLoader(<--->)
eval_dataloader = DataLoader(<--->)

#### **5. Training and Validation**

We have now gone through all the required preprocessing to prep the data for training. Instead of the Trainer module, it will be a good practice to initially write our own training loops so that we are mindful of all the steps that required for training neural networks.

Other than our training and validation data we need to select:


*   An optimizer to run backpropagation
*   A scheduler that sets a protocol for parameter updates at the end of a batch

We would also like to set a seed at the start of computation. This ensures that we are able to generate reproducicble results across multiple training sessions.

We run validation at the end of each epoch.



1.   ***Complete setting up the training loop and the validation after the end of an epoch.***
2.   ***Report the best validation loss obtained during training. We also want to save the model corresponding to the epoch that reported the best validation loss.***



In [ ]:
from transformers import get_linear_schedule_with_warmup
from tqdm.notebook import tqdm
from transformers import set_seed
from torch.optim import AdamW

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DistilBertForSequenceClassification.from_pretrained(name, num_labels=2).to(device)

num_epochs = 4
num_training_steps = len(train_dataloader)
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
lr_scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

best_val_loss = float("inf")
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
    # training
    model.train()
    training_losses = []
    for batch_i, batch in enumerate(train_dataloader):

        optimizer.zero_grad()

        # copy input to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Call the model for Forward Pass
        output = model(<--->)
        training_loss = output.loss
        training_losses.append(training_loss.item())

        #Do backprop and update params by taking an optimization step
        <--->
        <--->
        lr_scheduler.step()
        progress_bar.update(1)
    print("Mean Training Loss", np.mean(training_losses))

    # validation
    val_loss = 0
    #set to evaluation mode because we dont want to collect gradients
    <--->
    for batch_i, batch in enumerate(eval_dataloader):
        with <--->:
            # copy input to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            #call the model again for Forward Pass
            output = model(<--->)

        # add the batch average of validation loss to the running sum
        val_loss += output.loss

    # calculating average validation loss across all batches
    avg_val_loss = val_loss / len(eval_dataloader)
    print(f"Validation loss: {avg_val_loss}")

    # Saving this model checkpoint only if the current validation loss
    # is better than the best validation loss obtained so far
    if avg_val_loss < best_val_loss:
        print("Saving checkpoint!")
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': best_val_loss,
            },
            f"{model_path}epoch_{epoch}.pt"
        )
    print()

print(f"The best validation loss after {num_epochs} epochs is: {best_val_loss}")

#### **6. Evaluate your model on Test Data**

Now we use our finetuned model to evaluate the test set. We use performance metrics from `sklearn.metrics` to test the effectiveness of our model on unseen test data.

In order to do that, perform the following steps:

1. **Run the finetuned model you have just saved on your test data.**  
2. **Convert the logits returned by the model to class labels.**
3. **Report the following performance metrics:**
  *   ***Accuracy***
  *   ***F1 Score***



In [ ]:
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
eval_dataloader = DataLoader(tokenized_sst_dataset['validation'], batch_size=len(tokenized_sst_dataset['validation']))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.eval()
test_batch_logits = []
y_true = []
for batch_i, batch in enumerate(eval_dataloader):
    with torch.no_grad():
        # copy input to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().detach().numpy()

        # Call the model on test data
        output = model(<--->)
        test_batch_logits.append(output.logits)
        y_true.extend(labels)

In [ ]:
print(len(test_batch_logits),len(eval_dataloader))
test_logits = torch.cat(test_batch_logits, dim=0)

#sanity check -> dimension 0 of your logits tensor should be same as the size of the test dataset
print(test_logits.shape,len(tokenized_sst_dataset['validation']),len(y_true))

In [ ]:
#Convert the logits to predicted labels
y_pred = torch.argmax(<--->, dim = <--->).cpu().numpy()

print(y_true[:10])
print(y_pred[:10])

#sanity check: should have as many predictions as labels
assert len(y_pred)==len(y_true)

In [ ]:
# call the f1_score function
print('F1 Score:',f1_score(<--->))

# call the accuracy_score function
print('Accuracy Score:',accuracy_score(<--->))